# Task 4 - Improvement Cycle

The baseline evaluation from Task 3 showed that grounding was the weakest criterion, with multiple instances of unsupported claims and factual hallucinations.

To improve the results, I tested several changes aimed at making the generated descriptions more grounded while preserving fluency, tone, and length.

In [22]:
import os
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
BASE_DIR = Path.cwd().parent
SAMPLE_PATH = BASE_DIR / "outputs" / "assignment_01_sample.xlsx"

load_dotenv(BASE_DIR / ".env")

NEBIUS_API_KEY = os.getenv("NEBIUS_API_KEY")
if not NEBIUS_API_KEY:
    raise ValueError("Missing NEBIUS_API_KEY in environment variables.")

client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=NEBIUS_API_KEY,
)

GENERATION_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct-fast"

baseline_df = pd.read_excel(SAMPLE_PATH)
baseline_df.head()

,product_name,Product_attribute_list,material,warranty,user_prompt,generated_description,latency_ms,input_tokens,output_tokens,word_count,fluency,grammar,tone,length,grounding,latency_rating,cost_rating,final_score,error,cost_usd
0,Nintendo Switch OLED,"features: 7″ OLED screen, docked & handheld mo...",plastic chassis,1‑year limited warranty,Write a product description based on the follo...,Take your gaming experience to the next level ...,675.51,209,102,82,Good,Good,Ok,Good,Ok,Good,Good,FAIL,NaN,0.01545
1,Blink Outdoor 4,"features: 1080p HD, 2‑year battery, weather‑re...",plastic,1‑year limited warranty,Write a product description based on the follo...,"""Capture life's moments with the Blink Outdoor...",679.53,207,96,74,Good,Good,Good,Good,Ok,Good,Good,FAIL,NaN,0.01485
2,LEGO Star Wars Millennium Falcon 75192,"features: 7 541 pieces, detailed interior, min...",ABS plastic bricks,lifetime replacement of missing bricks,Write a product description based on the follo...,Get ready to embark on an epic adventure with ...,780.84,212,111,88,Good,Good,Good,Good,Bad,Good,Good,FAIL,NaN,0.01635
3,SanDisk Extreme PRO 128 GB SDXC,"features: UHS‑I U3, 200 MB/s read, waterproof;...",plastic,lifetime limited warranty,Write a product description based on the follo...,Capture life's most epic moments with the SanD...,676.98,212,93,72,Good,Good,Good,Good,Bad,Good,Good,FAIL,NaN,0.01473
4,Keurig K‑Supreme Plus Smart,"features: multistream extraction, BrewID, 78 o...",brushed stainless steel accents,1‑year limited warranty,Write a product description based on the follo...,"""Elevate your coffee game with the Keurig K-Su...",681.12,212,108,77,Good,Good,Ok,Good,Ok,Good,Good,FAIL,NaN,0.01608


In [3]:
def count_words(text: str) -> int:
    if not text:
        return 0
    return len(text.split())


def build_user_prompt(row: pd.Series) -> str:
    return f"""Write a product description based on the following product information:

product_name: {row['product_name']}
Product_attribute_list: {row['Product_attribute_list']}
material: {row['material']}
warranty: {row['warranty']}
""".strip()


def generate_description(user_prompt: str, system_prompt: str, model: str, temperature: float = 0.7) -> dict:
    start = time.perf_counter()

    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        max_tokens=180,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )

    end = time.perf_counter()

    generated_description = response.choices[0].message.content.strip()
    usage = response.usage

    return {
        "generated_description": generated_description,
        "latency_ms": round((end - start) * 1000, 2),
        "input_tokens": usage.prompt_tokens if usage else None,
        "output_tokens": usage.completion_tokens if usage else None,
        "word_count": count_words(generated_description),
    }

## Experiment 1 - Stricter Prompt

### What I changed
I introduced a stricter system prompt with explicit constraints to prevent unsupported claims and reduce exaggerated marketing language.

### Why I expected it to help
The baseline analysis showed that grounding was the weakest criterion. The model often introduced unsupported embellishments and factual hallucinations, so a stricter prompt should improve factual adherence.

In [16]:
STRICT_PROMPT_V2 = """
You are an expert e-commerce copywriter.

Write exactly one product description based only on the provided product data.

Requirements:
- Write in a friendly, credible, sales-oriented tone.
- The description must be between 50 and 90 words.
- Use clear, natural, fluent English.
- Use only facts explicitly stated in the input.
- Do not add benefits, qualities, or claims that are not directly supported by the input.
- Do not use exaggerated language such as "ultimate", "unparalleled", "best", "premium", "innovative", or similar unsupported promotional claims.
- If a detail is not explicitly given, do not mention it.
- Focus only on the listed features, material, and warranty.

IMPORTANT:
- Output only the final product description.
- Do not include any introductions, explanations, or labels.
- Do not write phrases like "Here is the description".
- Do not use line breaks before the text.
""".strip()

In [17]:
exp1_results = []

for _, row in baseline_df.iterrows():
    user_prompt = build_user_prompt(row)
    result = generate_description(
        user_prompt=user_prompt,
        system_prompt=STRICT_PROMPT_V2,
        model=GENERATION_MODEL,
        temperature=0.7,
    )

    combined = row.to_dict()
    combined["exp1_generated_description"] = result["generated_description"]
    combined["exp1_latency_ms"] = result["latency_ms"]
    combined["exp1_input_tokens"] = result["input_tokens"]
    combined["exp1_output_tokens"] = result["output_tokens"]
    combined["exp1_word_count"] = result["word_count"]

    exp1_results.append(combined)

pd.set_option("display.max_colwidth", None)
exp1_df = pd.DataFrame(exp1_results)
exp1_df[["product_name", "generated_description", "exp1_generated_description"]].head()

,product_name,generated_description,exp1_generated_description
0,Nintendo Switch OLED,"Take your gaming experience to the next level with the Nintendo Switch OLED. Boasting a vibrant 7″ OLED screen, this console offers stunning visuals in both docked and handheld modes. With 64 GB of storage, you'll have plenty of room for all your favorite games and apps. Built with a durable plastic chassis, this console is designed to last. Plus, enjoy peace of mind with our 1-year limited warranty. Get ready to immerse yourself in the world of gaming like never before.","Experience immersive gaming on the go with the Nintendo Switch OLED, featuring a stunning 7″ OLED screen and seamless transitions between docked and handheld modes. With 64 GB of storage, you'll have ample space to store your favourite games. The durable plastic chassis ensures a solid build, and for added peace of mind, your Switch is backed by a 1-year limited warranty."
1,Blink Outdoor 4,"""Capture life's moments with the Blink Outdoor 4, a limited edition camera that's as tough as it is innovative. Featuring 1080p HD video quality, this weather-resistant camera with a 2-year battery life is perfect for monitoring your home or office. Durable plastic construction and a sleek design make it a standout addition to any space. protected by a 1-year limited warranty, you can trust the Blink Outdoor 4 to keep an eye on things.""","The Blink Outdoor 4 is a weather-resistant security camera featuring 1080p HD video. Crafted from durable plastic, this camera is built to last. With a 2-year battery life, you can enjoy uninterrupted monitoring without worrying about frequent replacements. This limited edition model is backed by a 1-year limited warranty for added peace of mind."
2,LEGO Star Wars Millennium Falcon 75192,"Get ready to embark on an epic adventure with the LEGO Star Wars Millennium Falcon 75192! This highly detailed and iconic spaceship features an astonishing 7,541 pieces, including a detailed interior and a plethora of minifigures. Constructed with durable ABS plastic bricks, this model is built to last, and with its long-lasting battery, you'll be able to tackle even the most complex builds. And don't worry about losing a brick - with our lifetime warranty, you'll be covered for a replacement should any of your pieces go missing.","Bring the Star Wars universe to life with the LEGO Star Wars Millennium Falcon 75192. This incredible build features 7,541 pieces and a detailed interior, complete with numerous minifigures. Crafted from high-quality ABS plastic bricks, this model is built to last. In the event of any missing bricks, rest assured that we offer a lifetime replacement guarantee."
3,SanDisk Extreme PRO 128 GB SDXC,"Capture life's most epic moments with the SanDisk Extreme PRO 128 GB SDXC card, designed for professional photographers and adventurers alike. With UHS-I U3 speeds, you can transfer files at up to 200 MB/s, and its waterproof design ensures your memories are safe, even in the most challenging conditions. Built to last, this plastic-made card features long-lasting battery life, and is backed by a lifetime limited warranty for added peace of mind.","Capture life's adventures with the SanDisk Extreme PRO 128 GB SDXC card. Featuring UHS-I U3 speeds of up to 200 MB/s read, you can quickly transfer and store your photos and videos. With a long-lasting battery and waterproof design, this SD card is perfect for outdoor enthusiasts. Backed by a lifetime limited warranty, you can trust it to perform reliably."
4,Keurig K‑Supreme Plus Smart,"""Elevate your coffee game with the Keurig K-Supreme Plus Smart, a high-performance brewer designed to deliver rich and full-flavored cups. With multistream extraction and the innovative BrewID technology, you'll experience unparalleled flavor and convenience. The large 78oz reservoir allows for multiple brews without refill, and the brushed stainless steel accents add a touch of modern elegance to your countertop. Back

In [18]:
EXP1_V2_PATH = BASE_DIR / "outputs" / "assignment_01_exp1_v2.xlsx"

exp1_df.to_excel(EXP1_V2_PATH, index=False)

print(f"Saved to: {EXP1_V2_PATH}")

Saved to: /Users/mosheoz/Desktop/repos/llm-evaluation-pipeline/outputs/assignment_01_exp1_v2.xlsx


## Experiment 1 Results

The stricter prompt improved grounding in several cases by reducing unsupported promotional language and making the outputs more conservative.

For example, the Nintendo Switch OLED, Blink Outdoor 4, and Keurig K-Supreme Plus Smart descriptions became more factual and stayed closer to the provided input.

However, the improvement was not complete. In some cases, the model still introduced unsupported wording or inaccurate reformulations. For example, the LEGO Millennium Falcon description changed the warranty wording, and the SanDisk Extreme PRO description still included an incorrect battery-related claim.

Overall, this experiment improved grounding, but did not fully eliminate hallucinations.

---

## Experiment 2 - Lower Temperature

### What I changed
I reduced the temperature from 0.7 to 0.3 while keeping the stricter prompt.

### Why I expected it to help
A lower temperature should reduce creativity and make the model less likely to introduce unsupported claims.

In [19]:
exp2_results = []

for _, row in baseline_df.iterrows():
    user_prompt = build_user_prompt(row)
    result = generate_description(
        user_prompt=user_prompt,
        system_prompt=STRICT_PROMPT_V2,
        model=GENERATION_MODEL,
        temperature=0.3,
    )

    combined = row.to_dict()
    combined["exp2_generated_description"] = result["generated_description"]
    combined["exp2_latency_ms"] = result["latency_ms"]
    combined["exp2_input_tokens"] = result["input_tokens"]
    combined["exp2_output_tokens"] = result["output_tokens"]
    combined["exp2_word_count"] = result["word_count"]

    exp2_results.append(combined)

exp2_df = pd.DataFrame(exp2_results)
pd.set_option("display.max_colwidth", None)
exp2_df[[
    "product_name",
    "generated_description",
    "exp2_generated_description"
]].head()


,product_name,generated_description,exp2_generated_description
0,Nintendo Switch OLED,"Take your gaming experience to the next level with the Nintendo Switch OLED. Boasting a vibrant 7″ OLED screen, this console offers stunning visuals in both docked and handheld modes. With 64 GB of storage, you'll have plenty of room for all your favorite games and apps. Built with a durable plastic chassis, this console is designed to last. Plus, enjoy peace of mind with our 1-year limited warranty. Get ready to immerse yourself in the world of gaming like never before.","Experience gaming on the go with the Nintendo Switch OLED, featuring a 7″ OLED screen that delivers vibrant visuals. This versatile console offers both docked and handheld modes, allowing you to play your favorite games at home or on the go. With 64 GB of storage, you'll have ample space for your games and content. The console's durable plastic chassis ensures a long-lasting gaming experience. Enjoy your games with a 1-year limited warranty."
1,Blink Outdoor 4,"""Capture life's moments with the Blink Outdoor 4, a limited edition camera that's as tough as it is innovative. Featuring 1080p HD video quality, this weather-resistant camera with a 2-year battery life is perfect for monitoring your home or office. Durable plastic construction and a sleek design make it a standout addition to any space. protected by a 1-year limited warranty, you can trust the Blink Outdoor 4 to keep an eye on things.""","The Blink Outdoor 4 is a compact, weather-resistant security camera featuring 1080p HD video. Made from durable plastic, it's designed to withstand the elements. With a 2-year battery life, you can enjoy continuous monitoring without the need for frequent recharging. This limited edition model comes with a 1-year limited warranty for added peace of mind."
2,LEGO Star Wars Millennium Falcon 75192,"Get ready to embark on an epic adventure with the LEGO Star Wars Millennium Falcon 75192! This highly detailed and iconic spaceship features an astonishing 7,541 pieces, including a detailed interior and a plethora of minifigures. Constructed with durable ABS plastic bricks, this model is built to last, and with its long-lasting battery, you'll be able to tackle even the most complex builds. And don't worry about losing a brick - with our lifetime warranty, you'll be covered for a replacement should any of your pieces go missing.","Bring home the ultimate Star Wars experience with the LEGO Star Wars Millennium Falcon 75192. This highly detailed model features 7,541 pieces, including a detailed interior and a range of authentic minifigures. Built with long-lasting ABS plastic bricks, this iconic spaceship is sure to withstand countless play sessions. And with our lifetime warranty, you can rest assured that any missing bricks will be replaced for free."
3,SanDisk Extreme PRO 128 GB SDXC,"Capture life's most epic moments with the SanDisk Extreme PRO 128 GB SDXC card, designed for professional photographers and adventurers alike. With UHS-I U3 speeds, you can transfer files at up to 200 MB/s, and its waterproof design ensures your memories are safe, even in the most challenging conditions. Built to last, this plastic-made card features long-lasting battery life, and is backed by a lifetime limited warranty for added peace of mind.","Capture life's most epic moments with the SanDisk Extreme PRO 128 GB SDXC, a rugged and reliable storage solution. With UHS-I U3 speeds of up to 200 MB/s read, transfer your memories quickly and efficiently. Durable and waterproof, this SD card can withstand the toughest conditions. Built with long-lasting battery life, you can count on it to keep up with your active lifestyle. Backed by a lifetime limited warranty, you can trust that your memories are protected."
4,Keurig K‑Supreme Plus Smart,"""Elevate your coffee game with the Keurig K-Supreme Plus Smart, a high-performance brewer designed to deliver rich and full-flavored cups. With multistream extraction an

## Experiment 2 Results

Lowering the temperature from 0.7 to 0.3 significantly improved the consistency and factual accuracy of the outputs.

Compared to Experiment 1, the model became more conservative and adhered more closely to the provided input data. In several cases, this eliminated previous hallucinations, such as incorrect battery-related claims in the SanDisk example.

Grounding improved noticeably across most samples, with fewer unsupported claims and better alignment with the input.

However, this improvement came with a trade-off. The descriptions became less expressive and slightly less engaging, with a more neutral and factual tone.

Overall, reducing temperature proved to be an effective way to improve grounding, while slightly reducing the quality of the marketing tone.

---

## Experiment 3 - Larger Model + Lower Temperature

### What I changed
I switched to a larger model and reduced the temperature from 0.3 to 0.2, while keeping the stricter prompt from the previous experiments.

### Why I expected it to help
A larger model should improve reasoning and reduce hallucinations, while a lower temperature should further reduce randomness and unsupported claims.

In [20]:
LARGE_MODEL = "meta-llama/Llama-3.3-70B-Instruct"

In [ ]:
exp3_results = []

for _, row in baseline_df.iterrows():
    user_prompt = build_user_prompt(row)

    result = generate_description(
        user_prompt=user_prompt,
        system_prompt=STRICT_PROMPT_V2,
        model=LARGE_MODEL,
        temperature=0.2,
    )

    combined = row.to_dict()
    combined["exp3_generated_description"] = result["generated_description"]
    combined["exp3_latency_ms"] = result["latency_ms"]
    combined["exp3_input_tokens"] = result["input_tokens"]
    combined["exp3_output_tokens"] = result["output_tokens"]
    combined["exp3_word_count"] = result["word_count"]

    exp3_results.append(combined)

exp3_df = pd.DataFrame(exp3_results)
pd.set_option("display.max_colwidth", None)
exp3_df[["product_name", "generated_description", "exp3_generated_description"]].head()

,product_name,generated_description,exp3_generated_description
0,Nintendo Switch OLED,"Take your gaming experience to the next level with the Nintendo Switch OLED. Boasting a vibrant 7″ OLED screen, this console offers stunning visuals in both docked and handheld modes. With 64 GB of storage, you'll have plenty of room for all your favorite games and apps. Built with a durable plastic chassis, this console is designed to last. Plus, enjoy peace of mind with our 1-year limited warranty. Get ready to immerse yourself in the world of gaming like never before.","The Nintendo Switch OLED features a 7″ OLED screen and offers both docked and handheld modes, with 64 GB storage and a large capacity, all housed in a plastic chassis, backed by a 1‑year limited warranty."
1,Blink Outdoor 4,"""Capture life's moments with the Blink Outdoor 4, a limited edition camera that's as tough as it is innovative. Featuring 1080p HD video quality, this weather-resistant camera with a 2-year battery life is perfect for monitoring your home or office. Durable plastic construction and a sleek design make it a standout addition to any space. protected by a 1-year limited warranty, you can trust the Blink Outdoor 4 to keep an eye on things.""","The Blink Outdoor 4 features 1080p HD and is weather-resistant with a 2-year battery. Made of plastic, it's a limited edition product and comes with a 1-year limited warranty."
2,LEGO Star Wars Millennium Falcon 75192,"Get ready to embark on an epic adventure with the LEGO Star Wars Millennium Falcon 75192! This highly detailed and iconic spaceship features an astonishing 7,541 pieces, including a detailed interior and a plethora of minifigures. Constructed with durable ABS plastic bricks, this model is built to last, and with its long-lasting battery, you'll be able to tackle even the most complex builds. And don't worry about losing a brick - with our lifetime warranty, you'll be covered for a replacement should any of your pieces go missing.","The LEGO Star Wars Millennium Falcon 75192 is made of ABS plastic bricks and features 7,541 pieces, a detailed interior, and included minifigs, with a long-lasting battery and lifetime replacement of missing bricks."
3,SanDisk Extreme PRO 128 GB SDXC,"Capture life's most epic moments with the SanDisk Extreme PRO 128 GB SDXC card, designed for professional photographers and adventurers alike. With UHS-I U3 speeds, you can transfer files at up to 200 MB/s, and its waterproof design ensures your memories are safe, even in the most challenging conditions. Built to last, this plastic-made card features long-lasting battery life, and is backed by a lifetime limited warranty for added peace of mind.","The SanDisk Extreme PRO 128 GB SDXC features UHS‑I U3, 200 MB/s read, and is waterproof, with a long‑lasting battery, made of plastic, and backed by a lifetime limited warranty."
4,Keurig K‑Supreme Plus Smart,"""Elevate your coffee game with the Keurig K-Supreme Plus Smart, a high-performance brewer designed to deliver rich and full-flavored cups. With multistream extraction and the innovative BrewID technology, you'll experience unparalleled flavor and convenience. The large 78oz reservoir allows for multiple brews without refill, and the brushed stainless steel accents add a touch of modern elegance to your countertop. Backed by a 1-year limited warranty, you can trust this brewer to provide a delicious cup every time.""","The Keurig K-Supreme Plus Smart features multistream extraction and BrewID, with a large 78 oz reservoir. It has brushed stainless steel accents and comes with a 1-year limited warranty."


## Experiment 3 Results

Switching to a larger model improved consistency and made the outputs more conservative and closely aligned with the input data.

However, this came with several drawbacks. The generated descriptions became less engaging and more mechanical, resembling structured summaries rather than compelling product descriptions.

Additionally, the larger model did not fully eliminate hallucinations. In some cases, incorrect attributes (such as battery-related claims for products that do not have a battery) were still present.

The most significant downside was the substantial increase in latency and cost, making this approach less practical for real-time or large-scale applications.

Overall, while the larger model improved stability, it did not provide a meaningful advantage over the smaller model when combined with a well-designed prompt and lower temperature.

This demonstrates that increasing model size alone is not sufficient to fully solve hallucination issues, and that prompt design and decoding strategies play a more critical role in controlling model behavior.

---

---

## Final Conclusion

The most effective improvement came from combining a stricter prompt with a lower temperature.

This approach significantly improved grounding while maintaining a reasonable tone and low latency.

In contrast, switching to a larger model introduced higher cost and latency without providing a meaningful improvement in accuracy, demonstrating that prompt design and decoding parameters can be more impactful than model size.

This highlights the importance of prompt engineering and decoding strategies as primary levers for improving LLM performance in practical applications.